# When does the sigmoid become a step function — and why does it matter?

The steepness of the sigmoid is controlled by θ₁:

$$P(y=1 | x) = \frac{1}{1 + e^{-(\theta_0 + \theta_1 x)}}$$

As |θ₁| → ∞, the sigmoid approaches a **step function**: the model assigns probability 0 or 1 everywhere, with no uncertainty except exactly at the boundary.

We will show that this happens whenever the **training data is linearly separable** — and that this causes the training procedure to break down, regardless of why the data is separable.

## Setup: how the data is generated

We work with a **1D binary classification** problem. Each dataset has 10 points per class, drawn from two Gaussians with equal variance:

$$x \mid y=0 \sim \mathcal{N}\!\left(-\tfrac{\text{sep}}{2},\ \sigma^2\right), \qquad x \mid y=1 \sim \mathcal{N}\!\left(+\tfrac{\text{sep}}{2},\ \sigma^2\right)$$

Two parameters control the geometry:

- **sep**: distance between the two class means. Larger sep → classes further apart in x.
- **σ**: standard deviation of each class cluster. Larger σ → each class more spread out.

Their ratio is the **signal-to-noise ratio**:

$$\text{SNR} = \frac{\text{sep}}{\sigma}$$

and the overlap between the two Gaussians is:

$$\text{overlap} = \operatorname{erfc}\!\left(\frac{\text{sep}}{2\sigma\sqrt{2}}\right) \times 100\%$$

On each dataset we fit logistic regression, which learns θ₀ (boundary location) and θ₁ (sigmoid slope). The fitted θ₁ reflects both the separation and the spread of the training data — **not** just the SNR.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from scipy.special import erfc
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

def generate_data(n_per_class=10, class_sep=1.0, x_spread=1.0, seed=0):
    rng = np.random.RandomState(seed)
    X0  = rng.normal(loc=-class_sep/2, scale=x_spread, size=n_per_class)
    X1  = rng.normal(loc=+class_sep/2, scale=x_spread, size=n_per_class)
    X   = np.concatenate([X0, X1]).reshape(-1, 1)
    y   = np.array([0]*n_per_class + [1]*n_per_class)
    return X, y

def fit_logistic(X, y, C=1e4):
    clf = LogisticRegression(C=C, solver='lbfgs', max_iter=10000)
    clf.fit(X, y)
    return clf.intercept_[0], clf.coef_[0, 0]

def sigmoid(x, t0, t1):
    return 1 / (1 + np.exp(-np.clip(t0 + t1*x, -100, 100)))

def overlap_pct(sep, spread):
    return 100 * erfc(sep / (2 * spread * np.sqrt(2)))

def scatter_data(ax, X, y, seed):
    rng = np.random.RandomState(seed)
    j0  = rng.uniform(-0.04, 0.04, (y==0).sum())
    j1  = rng.uniform(-0.04, 0.04, (y==1).sum())
    ax.scatter(X[y==0], j0,     color='royalblue', s=50, alpha=0.9, zorder=3, label='class 0')
    ax.scatter(X[y==1], 1 + j1, color='tomato',    s=50, alpha=0.9, zorder=3, label='class 1')

x_plot = np.linspace(-12, 12, 500)
SEED   = 7
print("Ready.")

## Case 1 — Fixed separation, varying spread

We fix `sep = 3.0` and increase σ from 0.3 to 3.5. The SNR decreases across panels and the
classes overlap more and more.

For tight spread (σ=0.3) the training data is linearly separable and θ₁ is very large — the
sigmoid is nearly a step. As σ grows, the overlap increases, the data is no longer separable,
and the sigmoid smooths out.

This is the **normal** regime: θ₁ reflects the actual separability of the training data.

In [ ]:
SEP_FIXED = 3.0
spreads   = [0.3, 0.7, 1.2, 2.0, 3.5]

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
fig.suptitle(f'Fixed separation (sep={SEP_FIXED}), varying spread\n'
              'Tight spread → separable training data → step-like sigmoid',
             fontsize=12, fontweight='bold')

for ax, spread in zip(axes, spreads):
    X, y   = generate_data(10, SEP_FIXED, spread, seed=SEED)
    t0, t1 = fit_logistic(X, y)
    ov     = overlap_pct(SEP_FIXED, spread)
    snr    = SEP_FIXED / spread

    ax.plot(x_plot, sigmoid(x_plot, t0, t1), color='darkred', linewidth=2.2, zorder=2)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
    scatter_data(ax, X, y, seed=SEED + int(spread*10))

    ax.set_title(f'σ={spread},  SNR={snr:.1f}\noverlap={ov:.1f}%', fontsize=9)
    ax.annotate(f'θ₁={t1:.1f}', xy=(0.97, 0.52), xycoords='axes fraction',
                ha='right', fontsize=10, color='darkred', fontweight='bold')
    ax.set_xlim(-9, 9)
    ax.set_ylim(-0.15, 1.15)
    ax.set_yticks([0, 0.5, 1])
    ax.set_xlabel('x', fontsize=9)
    ax.grid(True, axis='x', alpha=0.25)

axes[0].set_ylabel('P(y=1|x)', fontsize=9)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig('case1_spread.png', dpi=130, bbox_inches='tight')
plt.show()

## Case 2 — Fixed SNR, varying spread

Now we keep `SNR = sep/σ = 10` constant by scaling sep with σ. The overlap is ~0% in every panel.

Yet θ₁ still decreases as σ grows. This is because θ₁ ∝ sep/σ² = SNR/σ — wider spread
attenuates the slope even when the separation scales with it. All training sets are separable,
but the resulting sigmoids range from nearly a step (small σ) to quite smooth (large σ).

The model with large σ is uncertain about where the boundary is — not because the classes
overlap, but because there is **no data near the boundary region**. Uncertainty about the
boundary location and uncertainty about class membership are two different things.

In [ ]:
SNR_FIXED = 10.0
spreads   = [0.3, 0.7, 1.2, 2.0, 3.5]

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
fig.suptitle(f'Fixed SNR={SNR_FIXED} (sep = SNR × σ), varying spread\n'
              'Same ~0% overlap in all panels — yet θ₁ decreases as σ grows',
             fontsize=12, fontweight='bold')

for ax, spread in zip(axes, spreads):
    sep    = SNR_FIXED * spread
    X, y   = generate_data(10, sep, spread, seed=SEED)
    t0, t1 = fit_logistic(X, y)
    ov     = overlap_pct(sep, spread)

    ax.plot(x_plot, sigmoid(x_plot, t0, t1), color='darkred', linewidth=2.2, zorder=2)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
    scatter_data(ax, X, y, seed=SEED + int(spread*10))

    ax.set_title(f'σ={spread},  sep={sep:.1f}\noverlap={ov:.2f}%', fontsize=9)
    ax.annotate(f'θ₁={t1:.1f}', xy=(0.97, 0.52), xycoords='axes fraction',
                ha='right', fontsize=10, color='darkred', fontweight='bold')
    ax.set_xlim(-12, 12)
    ax.set_ylim(-0.15, 1.15)
    ax.set_yticks([0, 0.5, 1])
    ax.set_xlabel('x', fontsize=9)
    ax.grid(True, axis='x', alpha=0.25)

axes[0].set_ylabel('P(y=1|x)', fontsize=9)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig('case2_fixed_snr.png', dpi=130, bbox_inches='tight')
plt.show()

## Why is the step-like sigmoid a problem?

The step-like sigmoid arises whenever the **training data is linearly separable** — there is a
clean gap between the two classes in x. This is a property of the training set alone, not of
any underlying distribution.

When the training data is separable, the cross-entropy loss has **no finite minimum**:

$$\mathcal{L}(\theta) = -\sum_{i} \Bigl[ y_i \log \sigma(\theta_0 + \theta_1 x_i)
                                        + (1-y_i) \log(1 - \sigma(\theta_0 + \theta_1 x_i)) \Bigr]$$

Making θ₁ larger always reduces the loss further — the optimizer has no reason to stop.
θ₁ → ∞ and training **never converges**.

This is not a statement about any population. The optimizer sees only the n training points.
If those points are separable, the MLE does not exist — full stop.

Consequences:

- **No convergence** — weights grow without bound
- **Overconfident predictions** — probability ≈ 0 or 1 everywhere, even in regions with no training data
- **Numerical instability** — log(0) terms in the loss

Note: the step function is not wrong as a *decision rule* when the classes are truly separated.
The problem is that the training procedure cannot reach it with finite parameters.

### The fix: L2 regularization

Adding an L2 penalty modifies the loss:

$$\mathcal{L}_{\text{reg}}(\theta) = \mathcal{L}(\theta) + \frac{1}{2C}\|\theta\|^2$$

The penalty grows with |θ₁|, creating a restoring force that pulls θ₁ back from infinity.
The loss now has a finite minimum — training converges. Smaller C = stronger regularization
= θ₁ pulled back more aggressively.

## Demonstration: regularization controls the steepness

In [ ]:
C_values = [1e4, 1.0, 0.1]
C_labels = ['C=10⁴  (no regularization)', 'C=1  (moderate)', 'C=0.1  (strong)']
colors_C = ['#d62728', '#e07b39', '#1f77b4']

# two separable datasets: tight vs wide spread, both SNR=10
cases = [
    (0.3,  3.0, 'Tight spread (σ=0.3, sep=3.0)\nSNR=10, training data separable'),
    (2.0, 20.0, 'Wide spread  (σ=2.0, sep=20.0)\nSNR=10, training data separable'),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
fig.suptitle('L2 regularization pulls θ₁ back from infinity\n'
             'Top: tight spread  |  Bottom: wide spread  |  Both training sets are separable',
             fontsize=12, fontweight='bold')

for r, (spread, sep, row_label) in enumerate(cases):
    X, y = generate_data(10, sep, spread, seed=SEED)
    for c, (C, cl, col) in enumerate(zip(C_values, C_labels, colors_C)):
        ax      = axes[r, c]
        t0, t1  = fit_logistic(X, y, C=C)

        ax.plot(x_plot, sigmoid(x_plot, t0, t1), color=col, linewidth=2.2, zorder=2)
        ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
        scatter_data(ax, X, y, seed=SEED + r)

        ax.annotate(f'θ₁={t1:.1f}', xy=(0.97, 0.52), xycoords='axes fraction',
                    ha='right', fontsize=10, color=col, fontweight='bold')
        ax.set_xlim(-12, 12)
        ax.set_ylim(-0.15, 1.15)
        ax.set_yticks([0, 0.5, 1])
        ax.grid(True, axis='x', alpha=0.25)
        ax.set_xlabel('x', fontsize=9)

        if r == 0:
            ax.set_title(cl, fontsize=10, color=col, fontweight='bold')
        if c == 0:
            ax.set_ylabel(row_label + '\nP(y=1|x)', fontsize=8)

plt.tight_layout()
plt.savefig('regularization_effect.png', dpi=130, bbox_inches='tight')
plt.show()